# Stability check (deterministic evaluation)

This notebook loads a saved EPPO checkpoint (eppo_best.pt) and runs a deterministic
argmax evaluation across many simulated students. It mirrors the `evaluate_checkpoint.py`
script but is structured as notebook cells you can run on Kaggle.

Instructions: upload `eppo_best.pt` to the notebook environment (e.g. via a Kaggle dataset),
set the `CKPT_PATH` variable below, then run all code cells.

In [ ]:
# Install lightweight dependencies (optional). On Kaggle these may already be present.
!pip install -q sentence-transformers scikit-learn

In [ ]:
import os
import sys
import json
import numpy as np
import torch
import importlib.util

# Try to locate and load pfa_eppo_poc.py from common Kaggle paths or the repo
def _find_and_load(fname='pfa_eppo_poc.py'):
    searched = []
    cwd = os.getcwd()
    candidates = [
        os.path.join(cwd, fname),
        os.path.join(cwd, 'Adaptive-Learning-Module', fname),
        os.path.join('/kaggle/working', fname),
    ]
    # search under /kaggle/input for uploaded dataset files
    kaggle_input = '/kaggle/input'
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            if fname in files:
                candidates.append(os.path.join(root, fname))
    for p in candidates:
        searched.append(p)
        if os.path.exists(p):
            spec = importlib.util.spec_from_file_location('pfa_eppo_poc', p)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod, searched
    return None, searched

mod, _searched = _find_and_load()
if mod is None:
    print('Could not find pfa_eppo_poc.py. Searched locations:')
    for s in _searched:
        print('  ', s)
    print('Please upload pfa_eppo_poc.py to /kaggle/working or add it as a Kaggle dataset under /kaggle/input.')
    raise ModuleNotFoundError('pfa_eppo_poc.py not found; upload it to the notebook environment')
# Export the names we need from the loaded module
Config = mod.Config
EPPOAgent = mod.EPPOAgent
PFATracker = mod.PFATracker
evaluate_policies = mod.evaluate_policies

# Configure these before running:
# On Kaggle, place eppo_best.pt inside /kaggle/working or use a dataset path under /kaggle/input/
CKPT_PATH = '/kaggle/working/eppo_best.pt'  # update as needed
N_STUDENTS = 1000
FAST_SIM = True  # True uses a synthetic sim-matrix (no model downloads)
OUT_JSON = 'evaluate_results.json'

cfg = Config()
DEVICE = cfg.DEVICE

In [ ]:
def tolerant_load(agent, path, device):
    ckpt = torch.load(path, map_location=device)
    try:
        agent.load_state_dict(ckpt)
        return True
    except Exception:
        pass
    candidates = [
        'model_state_dict', 'state_dict', 'actor_state_dict', 'actor',
        'agent_state_dict', 'policy_state_dict'
    ]
    for k in candidates:
        if isinstance(ckpt, dict) and k in ckpt:
            sub = ckpt[k]
            try:
                agent.load_state_dict(sub)
                return True
            except Exception:
                try:
                    if hasattr(agent, 'actor') and 'actor' in sub:
                        agent.actor.load_state_dict(sub.get('actor'))
                    if hasattr(agent, 'critic') and 'critic' in sub:
                        agent.critic.load_state_dict(sub.get('critic'))
                    return True
                except Exception:
                    continue
    if isinstance(ckpt, dict):
        try:
            if hasattr(agent, 'actor'):
                agent.actor.load_state_dict(ckpt)
                return True
        except Exception:
            pass
    return False

def build_sim_matrix(cfg, fast=True, seed=42):
    if fast:
        rng = np.random.default_rng(seed)
        emb = rng.normal(size=(cfg.N_CONCEPTS, 64))
        emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
        sim = emb @ emb.T
        return sim
    # full build via PFATracker (may download sentence-transformers)
    t = PFATracker(cfg.CONCEPTS, cfg)
    return t.sim_matrix

In [ ]:
# Run the deterministic evaluation
agent = EPPOAgent(cfg).to(DEVICE)
if not os.path.exists(CKPT_PATH):
    print('Checkpoint not found at', CKPT_PATH)
    raise FileNotFoundError(CKPT_PATH)
ok = tolerant_load(agent, CKPT_PATH, DEVICE)
if not ok:
    print('Failed to load checkpoint into agent; check format.')
    raise RuntimeError('Load failed')
sim_matrix = build_sim_matrix(cfg, fast=FAST_SIM)
policies = evaluate_policies(agent, cfg, sim_matrix, n_students=N_STUDENTS)
with open(OUT_JSON, 'w') as f:
    # Convert numpy types to native Python types where needed
    def _to_py(o):
        if isinstance(o, dict):
            return {k: _to_py(v) for k, v in o.items()}
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        if isinstance(o, np.ndarray):
            return o.tolist()
        return o
    json.dump({'ckpt': os.path.basename(CKPT_PATH), 'policies': _to_py(policies)}, f, indent=2)
print('Wrote results to', OUT_JSON)

# Print compact summary
for name, d in policies.items():
    try:
        goal = d.get('goal_rate', d.get('goal%') if isinstance(d, dict) else None)
        mean_apr = d.get('mean_apr', None)
        mean_steps = d.get('mean_steps', None)
    except Exception:
        goal = None; mean_apr = None; mean_steps = None
    if goal is not None:
        print(f"{name}: Goal%={goal:.1f}  Mean APR={mean_apr:.3f}  Mean steps={mean_steps:.1f}")
    else:
        print(f"{name}: (no summary available)")